# Ella2027 — LTX-2.5 22B Distilled on Google Colab (Run All)

دفتر الاستعادة الرسمي لمزوّد Ella2027. كل خطوات الإعداد موجودة هنا: فحص GPU والإنترنت، تثبيت WanGP مع الحفاظ على PyTorch الخاص بـ Google Colab، تنزيل ملفات LTX-2.5 المطلوبة أو إعادة استخدام الكاش، تسجيل ملف finetune المحلي، تنزيل عامل Ella2027 من هذا الفرع من GitHub، ثم تجهيز Real-ESRGAN x2 تلقائياً قبل تشغيل العامل.

**قبل Run All مرة واحدة:** فعّل GPU وInternet من إعدادات Google Colab، وأضف Google Colab Secret باسم `ELLA_COLAB_WORKER_TOKEN` بنفس قيمة `KAGGLE_LTX_WORKER_TOKEN` في Render. لا تضع التوكن في الدفتر أو GitHub.

يستخدم هذا الدفتر WanGP v20 عبر `shared.api` والنموذج `ltx2_25_22B_distilled`. أول تشغيل في Runtime جديد ينزّل الأوزان الكبيرة. تبقى حدود جلسات Google Colab والذاكرة وسرعة GPU قائمة.

## 1. Preflight — GPU, internet, and storage
Run this first. It does not download model weights.

In [ ]:
import os, shutil, torch, subprocess, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Google Colab GPU accelerator before continuing.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.2f} GB')
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
for path in ['/tmp', '/content']:
    u = shutil.disk_usage(path)
    print(f'{path}: free={u.free/1024**3:.1f} GB | total={u.total/1024**3:.1f} GB')
print('Model directory:', MODEL_DIR)
print('Preflight complete.')

## 2. Get WanGP source
The repository is cloned into the small working area. Model weights are stored separately in `/tmp`.

In [ ]:
import os, subprocess
WGP_DIR = '/content/WanGP'
if not os.path.isdir(os.path.join(WGP_DIR, '.git')):
    if os.path.exists(WGP_DIR):
        raise RuntimeError(f'{WGP_DIR} exists but is not a Git checkout. Rename/remove it before continuing.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/WanGP.git', WGP_DIR], check=True)
else:
    print('WanGP checkout already exists; keeping it unchanged.')
os.chdir(WGP_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
if not os.path.isfile(os.path.join(WGP_DIR, 'shared', 'api.py')):
    raise RuntimeError('This checkout does not include shared/api.py; the current WanGP API source is required.')
print('WanGP source and Python API ready:', WGP_DIR)


## 3. Install dependencies without replacing Google Colab's CUDA PyTorch
Pin the already-installed Torch family during requirements installation. This avoids an unnecessary large Torch/CUDA replacement.

In [ ]:
import os, sys, subprocess, torch
import importlib.util
from pathlib import Path
os.chdir(WGP_DIR)
constraints = Path('/tmp/ella-colab-constraints.txt')
pins = [f'torch=={torch.__version__.split("+")[0]}']
for package in ['torchvision', 'torchaudio']:
    if importlib.util.find_spec(package) is not None:
        mod = __import__(package)
        pins.append(f'{package}=={mod.__version__.split("+")[0]}')
constraints.write_text('\n'.join(pins) + '\n')
print('Constraints:', constraints.read_text())
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('WanGP requirements.txt was not found.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-c', str(constraints), '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'huggingface_hub>=0.30.0'], check=True)
print('Dependency installation complete. If Google Colab asks for a runtime restart, restart once and rerun cells from section 1.')

## 4. Download the two required LTX-2.5 checkpoints
Downloads are resumable through Hugging Face Hub's cache. If a download is interrupted, rerun this cell; completed files are reused. Do not save these large files in `/content`.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
main_name = 'ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors'
encoder_name = 'gemma4-12b-ltx-v1_int8_convrot.safetensors'
print('Downloading main LTX-2.5 checkpoint (large file; resumable)...')
main_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=main_name, local_dir=MODEL_DIR)
print('Main checkpoint ready:', main_path)
print('Downloading Gemma 4 12B INT8 ConvRot text encoder...')
encoder_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename='gemma4-12b-ltx-v1/' + encoder_name, local_dir=MODEL_DIR)
print('Text encoder ready:', encoder_path)
for p in [Path(main_path), Path(encoder_path)]:
    if not p.is_file() or p.stat().st_size < 100_000_000:
        raise RuntimeError(f'Checkpoint missing or unexpectedly small: {p}')
    print(f'VERIFIED {p.name}: {p.stat().st_size/1024**3:.2f} GB')
print('\nBoth required checkpoints are present.')

# Register the downloaded NVFP4 transformer and Gemma encoder as a local LTX-2.5 finetune.
# This is necessary because a checkpoint file in /tmp is not automatically selected by WanGP.
import json
finetunes_dir = Path(WGP_DIR) / "finetunes"
finetunes_dir.mkdir(parents=True, exist_ok=True)
finetune_profile = {
    "model": {
        "name": "LTX-2.5 22B Distilled NVFP4 — Ella local",
        "architecture": "ltx2_25_22B",
        "description": "Ella2027 local LTX-2.5 NVFP4 checkpoint and Gemma 4 INT8 ConvRot encoder.",
        "URLs": [str(Path(main_path).resolve())],
        "text_encoder_URLs": [str(Path(encoder_path).resolve())],
        "ltx2_pipeline": "distilled"
    }
}
finetune_path = finetunes_dir / "ltx2_25_22B_distilled.json"
finetune_path.write_text(json.dumps(finetune_profile, indent=2), encoding="utf-8")
print("Local LTX-2.5 finetune registered:", finetune_path)
print(finetune_path.read_text())


## 5. Prepare automatic Real-ESRGAN 2K enhancement

This section creates a separate Python 3.12 virtual environment under `/content/ella-python312`. It does **not** replace Google Colab's main Python, WanGP, or the Ella worker. The dependency combination below was verified on Google Colab Tesla T4: BasicSR 1.4.2, Real-ESRGAN 0.3.0, Torch 2.15.0+cu132, and TorchVision 0.29.2+cu132.

This preparation runs **before the worker starts**. It installs the isolated upscaler and downloads its x2 weights. Every new generated video is then enhanced automatically by the worker after generation; no manual cell is needed for new videos.

In [ ]:
import os, sys, shutil, subprocess, urllib.request
from pathlib import Path

VENV = Path('/content/ella-python312')
VENV_PY = VENV / 'bin' / 'python'
PY312 = shutil.which('python3.12') or '/usr/bin/python3.12'

if not Path(PY312).is_file():
    raise RuntimeError('Python 3.12 is not available in this Google Colab image; do not install into the main Python.')

if not VENV_PY.is_file():
    print('[ELLA] Creating isolated Python 3.12 environment...', flush=True)
    subprocess.run([PY312, '-m', 'venv', '--without-pip', str(VENV)], check=True)
else:
    print('[ELLA] Reusing existing isolated Python 3.12 environment.')

def pip_into_venv(args, label):
    print(f'[ELLA] {label}...', flush=True)
    p = subprocess.run(
        [sys.executable, '-m', 'pip', '--python', str(VENV_PY), 'install'] + args,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    if p.stdout:
        print(p.stdout[-3500:])
    if p.returncode:
        raise RuntimeError(f'{label} failed with exit code {p.returncode}')

pip_into_venv(['basicsr==1.4.2'], 'Installing BasicSR 1.4.2 in the isolated environment')
pip_into_venv(['--no-deps', 'realesrgan==0.3.0'], 'Installing Real-ESRGAN 0.3.0 without replacing Torch')

compat_test = r'''import sys, types, torch
from torchvision.transforms import functional as F
shim = types.ModuleType('torchvision.transforms.functional_tensor')
shim.rgb_to_grayscale = F.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = shim
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
print('BasicSR: OK')
print('Real-ESRGAN: OK')
print('Torch:', torch.__version__)
print('TorchVision:', __import__('torchvision').__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('CUDA is not available in the isolated environment')
print('GPU:', torch.cuda.get_device_name(0))
print('Upscaler environment verification: OK')'''

test = subprocess.run(
    [str(VENV_PY), '-c', compat_test],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(test.stdout)
if test.returncode:
    raise RuntimeError('Real-ESRGAN environment verification failed')

WEIGHTS = Path('/content/RealESRGAN_x2plus.pth')
if not WEIGHTS.is_file() or WEIGHTS.stat().st_size < 1_000_000:
    print('[ELLA] Downloading verified Real-ESRGAN x2 weights...', flush=True)
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        WEIGHTS,
    )
if not WEIGHTS.is_file() or WEIGHTS.stat().st_size < 1_000_000:
    raise RuntimeError('Real-ESRGAN x2 weights download is incomplete.')
print('[ELLA] Isolated Real-ESRGAN environment and x2 weights are ready.')

## 6. Start the Ella2027-connected WanGP worker

This starts WanGP's in-process Python API, keeps the model session warm, polls Ella2027 for video jobs, and sends completed MP4 files to the app's Cloudinary storage. The launcher starts the worker as a detached background process (`start_new_session=True`), so the notebook cell can finish and you may close the browser tab or switch apps after `WORKER STATUS: RUNNING` appears.

**Important:** closing the browser tab is different from stopping the Google Colab runtime. Do not choose Runtime → Disconnect and delete runtime, or terminate the session, while you need the GPU. Google Colab can still end a free session because of its own runtime, quota, or inactivity limits; no notebook code can guarantee an always-on GPU or override those limits.

The worker uses the existing Google Colab secret `ELLA_COLAB_WORKER_TOKEN` and the Render app URL `https://ella2027.onrender.com`.

In [ ]:
import os, sys, subprocess, requests, py_compile, time
from pathlib import Path
from google.colab import userdata

# Official worker source: always download the committed GitHub version.
WORKER_URL = "https://raw.githubusercontent.com/WaelNameh84/Ella2027/main/services/colab-ltx-worker/wangp_worker.py"
WORKER_PATH = Path("/content/ella_colab_wangp_worker.py")
WORKER_LOG = Path("/content/ella_colab_worker.log")
PID_FILE = Path("/content/ella_colab_worker.pid")

# If this exact worker was started earlier from this notebook, replace it cleanly.
if PID_FILE.exists():
    old_pid = PID_FILE.read_text().strip()
    if old_pid.isdigit():
        old_cmd = subprocess.getoutput(f"ps -p {old_pid} -o args=")
        if "ella_colab_wangp_worker.py" in old_cmd:
            subprocess.run(["kill", old_pid], check=False)
            for _ in range(10):
                if subprocess.run(["kill", "-0", old_pid], capture_output=True).returncode != 0:
                    break
                time.sleep(1)
            else:
                subprocess.run(["kill", "-9", old_pid], check=False)
            print(f"Stopped previous Ella worker PID {old_pid}.")

response = requests.get(WORKER_URL, timeout=90)
response.raise_for_status()
WORKER_PATH.write_text(response.text, encoding="utf-8")
py_compile.compile(str(WORKER_PATH), doraise=True)
print("Worker downloaded from GitHub and Python syntax check passed.")

token = userdata.get("ELLA_COLAB_WORKER_TOKEN")
if not token or not token.strip():
    raise RuntimeError("Google Colab Secret ELLA_COLAB_WORKER_TOKEN is missing or empty.")
os.environ["ELLA_COLAB_WORKER_TOKEN"] = token.strip()
os.environ["ELLA_API_BASE_URL"] = "https://ella2027.onrender.com"
os.environ["ELLA_WANGP_MODEL_TYPE"] = "ltx2_25_22B_distilled"

# Start detached so the notebook cell can finish while the worker keeps polling.
with WORKER_LOG.open("w", encoding="utf-8") as log_stream:
    proc = subprocess.Popen(
        [sys.executable, "-u", str(WORKER_PATH)],
        cwd="/content",
        env=os.environ.copy(),
        stdout=log_stream,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
PID_FILE.write_text(str(proc.pid), encoding="utf-8")
print("Worker PID:", proc.pid)
print("Worker log:", WORKER_LOG)

# Give startup a short window, then report whether the process survived.
time.sleep(25)
alive = subprocess.run(["kill", "-0", str(proc.pid)], capture_output=True).returncode == 0
print("WORKER STATUS:", "RUNNING" if alive else "EXITED")
if WORKER_LOG.exists():
    log_lines = WORKER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\\n========== WORKER STARTUP LOG ==========")
    print("\\n".join(log_lines[-70:]))
if not alive:
    raise RuntimeError("Ella2027 worker exited during startup. Read the worker log above; do not send video jobs yet.")
print("\\nWorker is running in the background. Keep this Google Colab session alive while generating videos.")

## 7. Verify and use Ella2027

The worker starts only after the Real-ESRGAN environment and weights are ready. It automatically enhances each generated video to 2x resolution before upload. The final setup cell stores its PID and log in `/content`.

1. Confirm the final cell prints `WORKER STATUS: RUNNING` and `Waiting for Ella2027 video jobs...`.
2. You may close the browser tab or leave Colab and use Ella2027; the detached worker continues while the Colab runtime itself remains alive.
3. Do not stop/delete the runtime. Colab may still stop a free runtime under its own limits; this setup cannot prevent provider-side shutdown.
4. Inspect `/content/ella_colab_worker.log` if needed. Do not launch a second worker while the recorded PID is alive.

## Recovery and runtime limits

- Import this exact notebook from `notebooks/Ella2027_Google_Colab_LTX_Worker_2K_Ready.ipynb` on the `main` branch. Enable GPU and Internet, add the Google Colab Secret `ELLA_COLAB_WORKER_TOKEN`, then use **Run All**.
- The worker is launched as a detached background process, so closing the browser tab does not by itself stop the worker. The Colab runtime must remain alive.
- A fresh Colab runtime must download WanGP source and model weights again. The worker source is always fetched from this branch. Never commit the worker token.
- The worker requires Render's `KAGGLE_LTX_WORKER_TOKEN` and the matching Colab secret.
- Google Colab is not an always-on host. Runtime duration, inactivity rules, GPU quotas, and provider-side shutdown remain in force. No code can guarantee that a free Colab GPU stays connected indefinitely.

## 8. Automatic 2x video enhancement

The full-video Real-ESRGAN x2 workflow was successfully tested on 2026-11-22. The sample output was 2048×1152 at 24 FPS, 5.041667 seconds, 6.59 MB, with an audio stream present. The original MP4 was preserved.

Use the optional manual cell in Section 9 when you intentionally want to upscale a completed video. It defaults to `RUN_FULL_2K = False`, so a normal **Run All** restores the worker and prepares the upscaler without starting expensive full-video processing. After the worker is healthy and GPU use is idle, change that setting to `True` and run the Section 9 cell deliberately.

Upscaling doubles pixel dimensions; it does not guarantee native 2K detail or flicker-free results for every video.

## 9. Automatic enhancement status

No manual action is required for new videos. The worker now performs this verified x2 pass automatically. This cell confirms the automatic mode and keeps Run All safe.

In [ ]:
print('[ELLA] Automatic Real-ESRGAN x2 enhancement is enabled for every new generated video.')
print('[ELLA] No manual upscaling cell is required. The worker preserves the original and uploads the enhanced MP4 when validation succeeds.')
